## Configuración Google Colab
> **Solo si estás en Colab:** ejecuta la siguiente celda para montar Google Drive.
> Si trabajas en local (VS Code / Jupyter), puedes saltártela.

In [1]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    # Change 'archive' to the name of the folder you uploaded to Drive
    DRIVE_PATH = '/content/drive/MyDrive/archive'
    os.chdir(DRIVE_PATH)
    print(f"Current directory: {os.getcwd()}")
    # Install dependencies
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'seaborn', 'scipy'])
else:
    import os
    print(f"Local environment — current directory: {os.getcwd()}")

Local environment — current directory: /home/josema/Proyectos/Bootcamp_IA/Proyectos/Proyecto4/linkedin


# EDA Proyecto I — Fase 2: Limpieza y Preparación
**DataTalent Solutions S.L.** | Módulo II: Análisis y Visualización de Datos

En esta fase **unimos los 11 CSV en un único DataFrame maestro** y aplicamos todas las técnicas de limpieza necesarias. Cada decisión está documentada con su justificación.

## 1. Importación de librerías

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded")

Libraries loaded


## 2. Carga de los 11 archivos CSV

In [3]:
postings             = pd.read_csv('postings.csv', low_memory=False)
companies            = pd.read_csv('companies/companies.csv', low_memory=False)
company_industries   = pd.read_csv('companies/company_industries.csv')
company_specialities = pd.read_csv('companies/company_specialities.csv')
employee_counts      = pd.read_csv('companies/employee_counts.csv')
benefits             = pd.read_csv('jobs/benefits.csv')
job_industries       = pd.read_csv('jobs/job_industries.csv')
job_skills           = pd.read_csv('jobs/job_skills.csv')
salaries             = pd.read_csv('jobs/salaries.csv')
industries           = pd.read_csv('mappings/industries.csv')
skills_map           = pd.read_csv('mappings/skills.csv')

print(f"11 CSVs loaded | postings: {postings.shape}")

11 CSVs loaded | postings: (123849, 31)


## 3. Unión (JOIN) de los 11 CSV en un DataFrame maestro

Construimos el DataFrame integrado siguiendo el esquema de relaciones detectado en la Fase 1.
Las tablas N:M (skills, industrias, beneficios, especialidades) se agregan como listas separadas por comas para mantener **una fila por oferta de empleo**.

In [4]:
# ── STEP 1: Salaries (1:1 per job_id) ────────────────────────────
sal = salaries[['job_id','max_salary','min_salary','med_salary',
                'pay_period','currency','compensation_type']].copy()
sal.columns = ['job_id','sal_max','sal_min','sal_med',
               'sal_period','sal_currency','sal_comp_type']

df = postings.merge(sal, on='job_id', how='left')
print(f"[1] + salaries:           {df.shape}")

# ── STEP 2: Companies (N:1 per company_id) ────────────────────────
comp = companies[['company_id','name','company_size',
                  'state','country','city']].copy()
comp.columns = ['company_id','comp_name','comp_size',
                'comp_state','comp_country','comp_city']
df = df.merge(comp, on='company_id', how='left')
print(f"[2] + companies:          {df.shape}")

# ── STEP 3: Employee Counts (N:1, most recent record) ─────────
emp = (employee_counts
       .sort_values('time_recorded', ascending=False)
       .drop_duplicates('company_id')
       [['company_id','employee_count','follower_count']])
df = df.merge(emp, on='company_id', how='left')
print(f"[3] + employee_counts:    {df.shape}")

# ── STEP 4: Company Industries (N:M → list) ───────────────────
ci_agg = (company_industries
          .groupby('company_id')['industry']
          .apply(lambda x: ', '.join(x.dropna().unique()))
          .reset_index()
          .rename(columns={'industry': 'comp_industries'}))
df = df.merge(ci_agg, on='company_id', how='left')
print(f"[4] + company_industries: {df.shape}")

# ── STEP 5: Company Specialities (N:M → list) ───────────────
cs_agg = (company_specialities
          .groupby('company_id')['speciality']
          .apply(lambda x: ', '.join(x.dropna().unique()))
          .reset_index()
          .rename(columns={'speciality': 'comp_specialities'}))
df = df.merge(cs_agg, on='company_id', how='left')
print(f"[5] + company_specialities:{df.shape}")

# ── STEP 6: Job Industries (N:M → list with name) ─────────
ji = job_industries.merge(industries, on='industry_id', how='left')
ji_agg = (ji.groupby('job_id')['industry_name']
            .apply(lambda x: ', '.join(x.dropna().unique()))
            .reset_index()
            .rename(columns={'industry_name': 'job_industries_list'}))
df = df.merge(ji_agg, on='job_id', how='left')
print(f"[6] + job_industries:     {df.shape}")

# ── STEP 7: Job Skills (N:M → list with full name) ────
js = job_skills.merge(skills_map, on='skill_abr', how='left')
js_agg = (js.groupby('job_id')['skill_name']
            .apply(lambda x: ', '.join(x.dropna().unique()))
            .reset_index()
            .rename(columns={'skill_name': 'job_skills_list'}))
df = df.merge(js_agg, on='job_id', how='left')
print(f"[7] + job_skills:         {df.shape}")

# ── STEP 8: Benefits (N:M → list) ──────────────────────────────
ben_agg = (benefits.groupby('job_id')['type']
            .apply(lambda x: ', '.join(x.dropna().unique()))
            .reset_index()
            .rename(columns={'type': 'benefits_list'}))
df = df.merge(ben_agg, on='job_id', how='left')
print(f"[8] + benefits:           {df.shape}")

print(f"\nMaster DataFrame: {df.shape[0]:,} rows x {df.shape[1]} columns")

[1] + salaries:           (123849, 37)
[2] + companies:          (123849, 42)
[3] + employee_counts:    (123849, 44)
[4] + company_industries: (123849, 45)
[5] + company_specialities:(123849, 46)
[6] + job_industries:     (123849, 47)
[7] + job_skills:         (123849, 48)
[8] + benefits:           (123849, 49)

Master DataFrame: 123,849 rows x 49 columns


In [5]:
print("Columns of the integrated master DataFrame:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:>2}. {col}")

Columns of the integrated master DataFrame:
   1. job_id
   2. company_name
   3. title
   4. description
   5. max_salary
   6. pay_period
   7. location
   8. company_id
   9. views
  10. med_salary
  11. min_salary
  12. formatted_work_type
  13. applies
  14. original_listed_time
  15. remote_allowed
  16. job_posting_url
  17. application_url
  18. application_type
  19. expiry
  20. closed_time
  21. formatted_experience_level
  22. skills_desc
  23. listed_time
  24. posting_domain
  25. sponsored
  26. work_type
  27. currency
  28. compensation_type
  29. normalized_salary
  30. zip_code
  31. fips
  32. sal_max
  33. sal_min
  34. sal_med
  35. sal_period
  36. sal_currency
  37. sal_comp_type
  38. comp_name
  39. comp_size
  40. comp_state
  41. comp_country
  42. comp_city
  43. employee_count
  44. follower_count
  45. comp_industries
  46. comp_specialities
  47. job_industries_list
  48. job_skills_list
  49. benefits_list


## Limpieza de Datos: Eliminación de Columnas Redundantes e Irrelevantes

Tras integrar los conjuntos de datos secundarios en el DataFrame maestro, varias columnas se volvieron redundantes (información duplicada) o irrelevantes para las tareas de análisis exploratorio de datos (EDA) y aprendizaje automático.

### Estrategia de Eliminación:
1. **Duplicados Directos:** Se eliminaron `comp_name` y `work_type` debido a que ya conservamos `company_name` y `formatted_work_type` (la cual contiene categorías más limpias y estandarizadas).
2. **Redundancia Salarial:** Las columnas originales sin procesar (`max_salary`, `min_salary`, etc.) se descartaron en favor del bloque estructurado `sal_*` creado durante la etapa de preprocesamiento.
3. **Metadatos y URLs de la Plataforma:** Se excluyeron los enlaces web como `job_posting_url`, `application_url` y `posting_domain`, ya que solo consumen memoria RAM y carecen de valor analítico.
4. **Geolocalización Ultra-específica:** Se eliminaron características de alta cardinalidad como `zip_code` y `fips` para evitar ruido en los modelos, manteniendo el análisis geográfico a nivel de ciudad, estado y país.

In [6]:
# List of columns to remove
columns_to_drop = [
    # Direct duplicates
    'comp_name', 'work_type',
    
    # Repeated salaries (keeping the clean 'sal_...' block)
    'max_salary', 'min_salary', 'med_salary', 'pay_period', 'currency', 'compensation_type',
    
    # URLs and links
    'job_posting_url', 'application_url', 'posting_domain',
    
    # Ultra-specific geolocation
    'zip_code', 'fips',
    
    # Timestamps (uncomment this line if you DO NOT need to analyze dates)
    # 'original_listed_time', 'listed_time', 'expiry', 'closed_time'
]

# Drop columns if they exist in the DataFrame
df_reduced = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

print(f"Columns before: {df.shape[1]} | Columns now: {df_reduced.shape[1]}")

Columns before: 49 | Columns now: 36


**Resultado:** Todos los 11 CSV están unidos en un único DataFrame maestro con una fila por oferta de empleo. Las columnas de tipo lista (skills, industrias, beneficios, especialidades) se han agregado como cadenas separadas por comas para mantener la integridad de la estructura tabular.

## 4. Filtrado: roles relacionados con datos
**Justificación:** El análisis se centra en roles de datos. Filtramos por palabras clave en el título para obtener un subconjunto relevante y reducir el ruido del análisis.

In [7]:
data_keywords = [
    'data', 'analyst', 'scientist', 'engineer', 'machine learning',
    'analytics', 'intelligence', 'statistician', 'quantitative',
    'etl', 'warehouse', 'big data', 'nlp', 'sql developer',
    'python developer', 'ai engineer', 'ml engineer'
]

mask = df['title'].str.lower().str.contains(
    '|'.join(data_keywords), na=False, regex=True
)
df_data = df[mask].copy()

print(f"Total job postings in the dataset:   {len(df):>8,}")
print(f"Data-related job roles:             {len(df_data):>8,}  ({len(df_data)/len(df)*100:.1f}%)")
print(f"\nTop 15 most frequent job titles in data roles:")
print(df_data['title'].value_counts().head(15).to_string())

Total job postings in the dataset:    123,849
Data-related job roles:               19,725  (15.9%)

Top 15 most frequent job titles in data roles:
title
Software Engineer                               181
Senior Software Engineer                        162
Business Analyst                                146
Data Analyst                                    137
Electrical Engineer                             122
Financial Analyst                               113
Package Handler - Part Time (Warehouse like)    112
Warehouse Associate                             103
Project Engineer                                102
Manufacturing Engineer                           97
Quality Engineer                                 92
Data Engineer                                    90
Mechanical Engineer                              87
Network Engineer                                 74
Board Certified Behavior Analyst (BCBA)          72


## 5. Normalización de textos (.str.lower, .str.strip, .str.replace)
**Justificación:** Sin normalización, "Data Engineer", "data engineer" y " Data Engineer " se tratan como 3 categorías distintas, inflando artificialmente la cardinalidad y distorsionando las frecuencias.

In [8]:
text_cols = ['title', 'company_name', 'location', 'comp_name',
             'comp_city', 'comp_country',
             'formatted_work_type', 'formatted_experience_level']

for col in text_cols:
    if col in df_data.columns:
        df_data[col] = (df_data[col]
                        .astype(str)
                        .str.lower()           # minúsculas
                        .str.strip()           # eliminar espacios extremos
                        .str.replace(r'\s+', ' ', regex=True)  # espacios múltiples
                        .replace('nan', np.nan))

print("Normalización aplicada: .lower() + .strip() + .replace(espacios)")
print("\nTítulos únicos (top 10) tras normalización:")
print(df_data['title'].value_counts().head(10).to_string())

Normalización aplicada: .lower() + .strip() + .replace(espacios)

Títulos únicos (top 10) tras normalización:
title
software engineer                               182
senior software engineer                        162
business analyst                                154
data analyst                                    137
electrical engineer                             122
financial analyst                               114
package handler - part time (warehouse like)    112
project engineer                                104
warehouse associate                             104
manufacturing engineer                           99


## 6. Normalización salarial — conversión a salario anual (USD)
El dataset mezcla salarios en distintas frecuencias de pago (hourly, monthly, yearly). Normalizamos todo a **salario anual**.

**Justificación:** Sin normalizar, un salario horario de $40 aparece idéntico a uno anual de $40, distorsionando por completo cualquier análisis comparativo. Las columnas `sal_*` provienen de `salaries.csv` (más completo); si no existen, usamos las de `postings.csv`.

In [9]:
HOURS_YEAR = 2080  # 40 h/week × 52 weeks

def annual_salary(row):
    # Priority: sal_med > sal_max > sal_min > fields from postings
    for field in ['sal_med', 'sal_max', 'sal_min', 'med_salary', 'max_salary', 'min_salary']:
        val = row.get(field)
        if not pd.isna(val):
            break
    else:
        return np.nan

    period = str(row.get('sal_period', row.get('pay_period', ''))).upper()
    if 'HOUR' in period:   return val * HOURS_YEAR
    if 'MONTH' in period:  return val * 12
    if 'WEEK'  in period:  return val * 52
    return val  # YEARLY or other -> assumed annual

df_data['salary_annual'] = df_data.apply(annual_salary, axis=1)

n_con = df_data['salary_annual'].notna().sum()
print(f"Records with calculated salary: {n_con:,}  ({n_con/len(df_data)*100:.1f}%)")
print(f"\nAnnual salary statistics (USD):")
print(df_data['salary_annual'].describe().round(0).to_string())

Records with calculated salary: 6,355  (32.2%)

Annual salary statistics (USD):
count        6355.0
mean       137882.0
std        226650.0
min             0.0
25%         90000.0
50%        124800.0
75%        166400.0
max      13665600.0


## 7. Tratamiento de valores nulos

| Variable | Estrategia | Justificación |
|----------|-----------|---------------|
| `formatted_experience_level` | Imputación con **moda** | Ordinal; el nivel más frecuente es representativo |
| `views`, `applies` | Imputación con **mediana** | Distribución sesgada → mediana es robusta |
| `company_name` / `comp_name` | Relleno con `'Desconocida'` | Evitar pérdida de filas con datos válidos en otras columnas |
| `salary_annual` | **Sin imputación** — análisis solo con registros reales | Inventar salarios distorsionaría las conclusiones y violaría la integridad analítica |
| Columnas con >85% nulos | Excluidas del análisis estadístico principal | No aportan información suficiente para inferencias válidas |

In [10]:
# 1. Experience level → mode
mode_exp_s = df_data['formatted_experience_level'].mode()
mode_exp   = mode_exp_s[0] if len(mode_exp_s) > 0 else 'mid-senior level'
df_data['formatted_experience_level'] = df_data['formatted_experience_level'].fillna(mode_exp)
print(f"experience_level → imputed with mode: '{mode_exp}'")

# 2. Views and applies → median
for col in ['views', 'applies']:
    med = df_data[col].median()
    df_data[col] = df_data[col].fillna(med)
    print(f"{col} → imputed with median: {med:.0f}")

# 3. Company name
df_data['company_name'] = df_data['company_name'].fillna('Unknown')

# 4. Subset with salary (for salary analysis)
df_sal = df_data[df_data['salary_annual'].notna()].copy()
print(f"\nRecords with available salary: {len(df_sal):,} ({len(df_sal)/len(df_data)*100:.1f}%)")

# Verify result
print("\nRemaining nulls in key columns:")
cols_check = ['formatted_experience_level', 'views', 'applies', 'company_name', 'salary_annual']
for c in cols_check:
    if c in df_data.columns:
        n = df_data[c].isna().sum()
        print(f"  {c}: {n}")

experience_level → imputed with mode: 'mid-senior level'
views → imputed with median: 5
applies → imputed with median: 4

Records with available salary: 6,355 (32.2%)

Remaining nulls in key columns:
  formatted_experience_level: 0
  views: 0
  applies: 0
  company_name: 0
  salary_annual: 13370


## 8. Detección de outliers en salario

Usamos dos métodos complementarios:
- **IQR (Rango Intercuartílico):** robusto, no asume distribución
- **Z-score:** basado en la distancia a la media en desviaciones estándar

In [11]:
salaries = df_sal['salary_annual'].dropna()

# ── Method 1: IQR ─────────────────────────────────────────────────
Q1  = salaries.quantile(0.25)
Q3  = salaries.quantile(0.75)
IQR = Q3 - Q1
lower_lim = max(Q1 - 1.5 * IQR, 10_000)   # reasonable minimum: $10k/year
upper_lim = Q3 + 1.5 * IQR

out_iqr = df_sal[(df_sal['salary_annual'] < lower_lim) |
                 (df_sal['salary_annual'] > upper_lim)]

print("=== IQR Method ===")
print(f"  Q1  (P25) = ${Q1:>12,.0f}")
print(f"  Q3  (P75) = ${Q3:>12,.0f}")
print(f"  IQR       = ${IQR:>12,.0f}")
print(f"  Lower limit: ${lower_lim:>10,.0f}")
print(f"  Upper limit: ${upper_lim:>10,.0f}")
print(f"  IQR Outliers: {len(out_iqr):,}  ({len(out_iqr)/len(df_sal)*100:.1f}%)")

=== IQR Method ===
  Q1  (P25) = $      90,000
  Q3  (P75) = $     166,400
  IQR       = $      76,400
  Lower limit: $    10,000
  Upper limit: $   281,000
  IQR Outliers: 247  (3.9%)


In [12]:
# ── Method 2: Z-score ─────────────────────────────────────────────
z_scores = np.abs(stats.zscore(df_sal['salary_annual']))
out_z = df_sal[z_scores > 3]

print("=== Z-score Method (threshold ±3σ) ===")
print(f"  Mean = ${df_sal['salary_annual'].mean():>10,.0f}")
print(f"  Std  = ${df_sal['salary_annual'].std():>10,.0f}")
print(f"  Outliers (|z| > 3): {len(out_z):,}  ({len(out_z)/len(df_sal)*100:.1f}%)")
print()
print("Most extreme salaries (top 5):")
print(df_sal.nlargest(5, 'salary_annual')[['title', 'comp_name', 'salary_annual']].to_string(index=False))

=== Z-score Method (threshold ±3σ) ===
  Mean = $   137,882
  Std  = $   226,650
  Outliers (|z| > 3): 8  (0.1%)

Most extreme salaries (top 5):
                               title        comp_name  salary_annual
                       data engineer          koantek     13665600.0
                  wifi test engineer    quantumbridge     10524800.0
neo4j- senior database administrator       sstech llc      1300000.0
             quantitative researcher goliath partners      1200000.0
             quantitative researcher goliath partners      1000000.0


**Decisiones documentadas:**

| Tipo de outlier | Criterio | Acción | Justificación |
|----------------|---------|--------|---------------|
| Salario < $10.000/año | Umbral mínimo | **Eliminar** | Probablemente un salario horario mal registrado como anual |
| Salario > límite IQR superior | Método IQR | **Eliminar** | Valores inverosímiles; errores de captura |
| Salarios altos legítimos dentro del IQR | — | **Conservar** | Roles directivos o en FAANG justifican salarios elevados reales |

In [13]:
# Apply outlier filter
df_clean = df_sal[
    (df_sal['salary_annual'] >= lower_lim) &
    (df_sal['salary_annual'] <= upper_lim)
].copy()

print(f"With salary (before filtering outliers): {len(df_sal):>7,}")
print(f"With salary (after filtering):          {len(df_clean):>7,}")
print(f"Outliers removed:                        {len(df_sal)-len(df_clean):>7,}")
print(f"\nClean salary range: ${df_clean['salary_annual'].min():,.0f} — ${df_clean['salary_annual'].max():,.0f}")

With salary (before filtering outliers):   6,355
With salary (after filtering):            6,108
Outliers removed:                            247

Clean salary range: $13,333 — $281,000


## 9. Guardado de datasets procesados

In [14]:
# Full master dataset (all roles, all joins)
df.to_csv('data_maestro_completo.csv', index=False)
print(f"Saved: data_maestro_completo.csv     — {df.shape}")

# Dataset filtered to data roles (all, without salary restrictions)
df_data.to_csv('data_roles_completo.csv', index=False)
print(f"Saved: data_roles_completo.csv       — {df_data.shape}")

# Dataset of data roles WITH clean salary (for salary analysis)
df_clean.to_csv('data_roles_salario.csv', index=False)
print(f"Saved: data_roles_salario.csv        — {df_clean.shape}")

Saved: data_maestro_completo.csv     — (123849, 49)
Saved: data_roles_completo.csv       — (19725, 50)
Saved: data_roles_salario.csv        — (6108, 50)


## 10. Resumen de decisiones de limpieza

| Paso | Acción | Justificación |
|------|--------|---------------|
| **Joins** | 11 CSV unidos → 1 DataFrame maestro | Análisis integrado con toda la información disponible |
| **Filtro roles** | Keywords en `title` | Enfocar el análisis en roles de datos |
| **Normalización texto** | `.lower()` + `.strip()` + `.replace()` | Estandarizar categorías para evitar duplicados por capitalización |
| **Salario anual** | Unificación de unidades (hourly→anual, monthly→anual) | Comparar salarios en la misma unidad |
| **experience_level** | Imputación con moda | Categoría ordinal; el nivel más común es representativo |
| **views / applies** | Imputación con mediana | Robustez frente a distribución asimétrica |
| **salary_annual** | Sin imputar; análisis solo con datos reales | Inventar salarios genera conclusiones falsas |
| **Outliers salariales** | Filtro IQR + umbral mínimo $10k | Eliminar errores de entrada; conservar casos legítimos |

**Próximo paso (Fase 3):** Análisis estadístico descriptivo, correlaciones, groupby y detección de sesgos.